# Notebook 2: Endometrial Cancer DEG Analysis (TCGA-UCEC)
## CausalPMOS-Map — Cancer Evidence C(g)

**Goal:** Identify genes differentially expressed in endometrial cancer vs normal endometrium. These feed the **Cancer component C(g)** of the PCBS score.

**Dataset:** [TCGA-UCEC](https://portal.gdc.cancer.gov/projects/TCGA-UCEC) (The Cancer Genome Atlas — Uterine Corpus Endometrial Carcinoma)  
- ~560 primary tumour + ~23 solid-tissue normal samples  
- RNA-Seq (Illumina HiSeq), pan-cancer normalised log₂(RSEM + 1)  
- Rich clinical metadata: grade, stage, histological subtype  

**Methods:**
- **limma** with empirical Bayes (`trend = TRUE` for RNA-seq)  
- Adjusted p-value < 0.05 only — **no hard log₂FC cutoff**  
- log₂FC saved as continuous weight for downstream PCBS C(g)  

**Key output:** `results/all_genes_limma_tcga_ucec.csv`

## 1. Setup

In [ ]:
# ── 1 · Install packages & imports ─────────────────────────────
!pip install -q rpy2 gseapy adjustText

import warnings, os
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from io import BytesIO

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('Setup complete.')

In [ ]:
# ── Install limma in R via rpy2 ────────────────────────────────
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
pandas2ri.activate()

ro.r('''
if (!requireNamespace("BiocManager", quietly = TRUE))
    install.packages("BiocManager", repos = "https://cloud.r-project.org")
BiocManager::install("limma", update = FALSE, ask = FALSE)
library(limma)
cat("limma", as.character(packageVersion("limma")), "loaded\\n")
''')

## 2. Download TCGA-UCEC Data from UCSC Xena

We pull pre-processed RNA-Seq expression and clinical data from the
[UCSC Xena TCGA hub](https://xenabrowser.net/datapages/?cohort=TCGA%20Uterine%20Corpus%20Endometrial%20Carcinoma%20(UCEC)).  
Expression values are **log₂(RSEM + 1)**, already normalised — no probe mapping needed (gene symbols are row IDs).

In [ ]:
# ── 2 · Download expression + clinical matrices ─────────────────

URL_EXPR = (
    'https://tcga-xena-hub.s3.us-east-1.amazonaws.com/'
    'download/TCGA.UCEC.sampleMap%2FHiSeqV2.gz'
)
URL_CLIN = (
    'https://tcga-xena-hub.s3.us-east-1.amazonaws.com/'
    'download/TCGA.UCEC.sampleMap%2FUCEC_clinicalMatrix'
)

def _download(url, label):
    print(f'  Downloading {label} …', end=' ')
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    print(f'{len(r.content)/1e6:.1f} MB')
    return BytesIO(r.content)

try:
    expr_raw = pd.read_csv(
        _download(URL_EXPR, 'gene expression'),
        sep='\t', index_col=0, compression='gzip'
    )
    clin = pd.read_csv(
        _download(URL_CLIN, 'clinical data'),
        sep='\t', index_col=0
    )
except Exception as e:
    raise RuntimeError(
        f'Download failed ({e}).\n'
        'Fallback: visit https://xenabrowser.net/datapages/ ,\n'
        'search "TCGA.UCEC.sampleMap/HiSeqV2", download both files,\n'
        'and load them with pd.read_csv() manually.'
    )

print(f'\nExpression : {expr_raw.shape[0]:,} genes × {expr_raw.shape[1]} samples')
print(f'Clinical   : {clin.shape[0]} entries × {clin.shape[1]} fields')
print(f'\nFirst 5 gene symbols: {list(expr_raw.index[:5])}')
print(f'First 3 sample IDs  : {list(expr_raw.columns[:3])}')

## 3. Parse Clinical Metadata

TCGA barcodes encode sample type in positions 13-14:  
- `01` → Primary Tumour  
- `11` → Solid Tissue Normal

In [ ]:
# ── 3 · Classify samples + extract grade ───────────────────────

def tcga_sample_type(barcode):
    parts = barcode.split('-')
    if len(parts) >= 4:
        code = parts[3][:2]
        if code == '01': return 'Tumor'
        if code == '11': return 'Normal'
    return 'Other'

sample_meta = pd.DataFrame({
    'barcode': expr_raw.columns,
    'sample_type': [tcga_sample_type(b) for b in expr_raw.columns],
    'patient_id':  [b[:12] for b in expr_raw.columns],
})

# ── Match with clinical data (try 15-char then 12-char key) ──
clin = clin.copy()
clin['_key15'] = clin.index.astype(str).str[:15]
clin['_key12'] = clin.index.astype(str).str[:12]

# Find grade column
grade_col = next(
    (c for c in clin.columns if 'grade' in c.lower()),
    None
)

# Find histology column
hist_col = next(
    (c for c in clin.columns if 'histolog' in c.lower() and 'type' in c.lower()),
    None
)

# Map grade to each sample
if grade_col:
    key15_to_grade = dict(zip(clin['_key15'], clin[grade_col].astype(str)))
    key12_to_grade = dict(zip(clin['_key12'], clin[grade_col].astype(str)))
    sample_meta['grade_raw'] = sample_meta['barcode'].apply(
        lambda b: key15_to_grade.get(b[:15], key12_to_grade.get(b[:12], 'Unknown'))
    )
    # Normalise grade labels → G1 / G2 / G3 / High Grade / Unknown
    def normalise_grade(g):
        g = str(g).strip().upper()
        if g in ('G1', '1', 'GRADE 1'): return 'G1'
        if g in ('G2', '2', 'GRADE 2'): return 'G2'
        if g in ('G3', '3', 'GRADE 3'): return 'G3'
        if 'HIGH' in g:                 return 'High Grade'
        return 'Unknown'
    sample_meta['grade'] = sample_meta['grade_raw'].apply(normalise_grade)
else:
    sample_meta['grade'] = 'Unknown'

# Map histology
if hist_col:
    key15_to_hist = dict(zip(clin['_key15'], clin[hist_col].astype(str)))
    key12_to_hist = dict(zip(clin['_key12'], clin[hist_col].astype(str)))
    sample_meta['histology'] = sample_meta['barcode'].apply(
        lambda b: key15_to_hist.get(b[:15], key12_to_hist.get(b[:12], 'Unknown'))
    )
else:
    sample_meta['histology'] = 'Unknown'

# ── Summary ──
tumor_ids  = sample_meta.loc[sample_meta['sample_type'] == 'Tumor',  'barcode'].tolist()
normal_ids = sample_meta.loc[sample_meta['sample_type'] == 'Normal', 'barcode'].tolist()

print(f'Sample types:')
print(sample_meta['sample_type'].value_counts().to_string())
print(f'\nTumour grades (tumour samples only):')
print(sample_meta.loc[sample_meta['sample_type']=='Tumor', 'grade'].value_counts().to_string())
if hist_col:
    print(f'\nHistological subtypes (tumour only):')
    print(sample_meta.loc[sample_meta['sample_type']=='Tumor', 'histology'].value_counts().head(10).to_string())
print(f'\n→ Tumour: {len(tumor_ids)}   Normal: {len(normal_ids)}')

## 4. Quality Control

In [ ]:
# ── 4a · Filter low-expression genes ───────────────────────────
# Keep genes expressed (> 0) in at least 10 % of samples
min_samples = int(0.10 * expr_raw.shape[1])
gene_expressed = (expr_raw > 0).sum(axis=1) >= min_samples
expr = expr_raw.loc[gene_expressed].copy()

print(f'Genes before filter: {expr_raw.shape[0]:,}')
print(f'Genes after filter : {expr.shape[0]:,}  (expressed in ≥ {min_samples} samples)')

In [ ]:
# ── 4b · Expression distribution (Tumour vs Normal) ────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Density of median gene expression
for grp, ids, col in [('Tumor', tumor_ids, '#e74c3c'),
                       ('Normal', normal_ids, '#3498db')]:
    medians = expr[ids].median(axis=1)
    axes[0].hist(medians, bins=80, alpha=0.5, color=col, label=grp, density=True)
axes[0].set_xlabel('Median log₂(RSEM+1)')
axes[0].set_ylabel('Density')
axes[0].set_title('Gene-level median expression')
axes[0].legend()

# PCA
from sklearn.decomposition import PCA
all_ids = tumor_ids + normal_ids
X = expr[all_ids].T.values
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)
labels = ['Tumor']*len(tumor_ids) + ['Normal']*len(normal_ids)
colors = ['#e74c3c' if l=='Tumor' else '#3498db' for l in labels]
axes[1].scatter(coords[:,0], coords[:,1], c=colors, s=15, alpha=0.6)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title('PCA — Tumour (red) vs Normal (blue)')

plt.tight_layout()
plt.savefig('figures/nb2_01_qc.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance')

## 5. limma DEG Analysis — Tumour vs Normal

We use **limma** with `eBayes(trend = TRUE)`, recommended for log-transformed RNA-seq data.  
DEGs are called at **adj. p-value < 0.05 only** — no log₂FC cutoff.  
The continuous log₂FC is retained for PCBS C(g) weighting.

In [ ]:
# ── 5 · limma helper (reusable) ────────────────────────────────

def run_limma(expr_df, group1_ids, group2_ids,
              group1_name='Tumor', group2_name='Normal'):
    """
    Run limma eBayes (trend=TRUE) on log-transformed expression.
    Returns DataFrame with logFC, adj.P.Val, Direction for every gene.
    Contrast: group1 − group2  (positive logFC ⇒ higher in group1).
    """
    samples = [s for s in group1_ids + group2_ids if s in expr_df.columns]
    g1 = [s for s in group1_ids if s in expr_df.columns]
    g2 = [s for s in group2_ids if s in expr_df.columns]
    print(f'  limma: {len(g1)} {group1_name} vs {len(g2)} {group2_name}')

    expr_sub = expr_df[g1 + g2]
    groups = [group1_name] * len(g1) + [group2_name] * len(g2)

    # Transfer to R
    ro.globalenv['expr_mat']   = pandas2ri.py2rpy(expr_sub)
    ro.globalenv['grp']        = ro.StrVector(groups)
    ro.globalenv['g1_name']    = group1_name
    ro.globalenv['g2_name']    = group2_name

    ro.r('''
    library(limma)
    grp   <- factor(grp, levels = c(g2_name, g1_name))
    design <- model.matrix(~ 0 + grp)
    colnames(design) <- levels(grp)

    fit  <- lmFit(as.matrix(expr_mat), design)
    cont <- makeContrasts(contrasts = paste0(g1_name, " - ", g2_name),
                          levels = design)
    fit2 <- contrasts.fit(fit, cont)
    fit2 <- eBayes(fit2, trend = TRUE)

    res  <- topTable(fit2, number = Inf, sort.by = "none")
    ''')

    results = pandas2ri.rpy2py(ro.globalenv['res'])
    results.index = expr_sub.index
    results = results.reset_index().rename(columns={'index': 'Gene'})

    # Standardise column names
    col_map = {'adj.P.Val': 'adj_pvalue', 'P.Value': 'pvalue',
               'AveExpr': 'AveExpr', 'logFC': 'logFC', 't': 't_stat', 'B': 'B'}
    results = results.rename(columns=col_map)

    # Direction: adj. p < 0.05, sign of logFC
    results['Direction'] = 'NS'
    sig = results['adj_pvalue'] < 0.05
    results.loc[sig & (results['logFC'] > 0), 'Direction'] = 'Up'
    results.loc[sig & (results['logFC'] < 0), 'Direction'] = 'Down'

    n_up   = (results['Direction'] == 'Up').sum()
    n_down = (results['Direction'] == 'Down').sum()
    print(f'  → {n_up} Up + {n_down} Down = {n_up+n_down} DEGs  '
          f'(adj.p < 0.05, no logFC cutoff)')
    return results

In [ ]:
# ── Run primary analysis: Tumour vs Normal ─────────────────────
print('═' * 55)
print('PRIMARY: Tumour vs Normal (all grades)')
print('═' * 55)
deg_cancer = run_limma(expr, tumor_ids, normal_ids, 'Tumor', 'Normal')

# Save primary output → this feeds PCBS C(g)
deg_cancer.to_csv('results/all_genes_limma_tcga_ucec.csv', index=False)
print(f'\nSaved: results/all_genes_limma_tcga_ucec.csv  ({len(deg_cancer)} genes)')

# Top 20 by absolute logFC among significant genes
sig = deg_cancer[deg_cancer['Direction'] != 'NS'].copy()
sig['abs_logFC'] = sig['logFC'].abs()
print(f'\nTop 20 DEGs by |logFC|:')
print(sig.nlargest(20, 'abs_logFC')[['Gene','logFC','adj_pvalue','Direction']].to_string(index=False))

## 6. Grade-Specific Analyses

Understanding how gene expression changes across tumour grades helps characterise the PMOS → cancer progression:
- **Grade 1 vs Normal** — earliest cancer changes  
- **Grade 3 vs Grade 1** — genes driving aggressive progression

In [ ]:
# ── 6 · Grade-specific DEG ─────────────────────────────────────

g1_ids = sample_meta.loc[
    (sample_meta['sample_type']=='Tumor') & (sample_meta['grade']=='G1'),
    'barcode'
].tolist()
g2_ids = sample_meta.loc[
    (sample_meta['sample_type']=='Tumor') & (sample_meta['grade']=='G2'),
    'barcode'
].tolist()
g3_ids = sample_meta.loc[
    (sample_meta['sample_type']=='Tumor') & (sample_meta['grade']=='G3'),
    'barcode'
].tolist()

print(f'Grade 1: {len(g1_ids)}   Grade 2: {len(g2_ids)}   Grade 3: {len(g3_ids)}')
print(f'Normal : {len(normal_ids)}\n')

# Grade 1 vs Normal
deg_g1_vs_normal = None
if len(g1_ids) >= 5 and len(normal_ids) >= 5:
    print('─' * 45)
    print('Grade 1 vs Normal (early cancer)')
    deg_g1_vs_normal = run_limma(expr, g1_ids, normal_ids, 'Grade1', 'Normal')
    deg_g1_vs_normal.to_csv('results/deg_grade1_vs_normal_tcga_ucec.csv', index=False)

# Grade 3 vs Grade 1 (progression)
deg_g3_vs_g1 = None
if len(g3_ids) >= 5 and len(g1_ids) >= 5:
    print('\n' + '─' * 45)
    print('Grade 3 vs Grade 1 (aggressive progression)')
    deg_g3_vs_g1 = run_limma(expr, g3_ids, g1_ids, 'Grade3', 'Grade1')
    deg_g3_vs_g1.to_csv('results/deg_grade3_vs_grade1_tcga_ucec.csv', index=False)

## 7. Volcano Plots

In [ ]:
# ── 7 · Volcano plots ──────────────────────────────────────────

def volcano(df, title, ax, top_n=10):
    palette = {'Up': '#e74c3c', 'Down': '#3498db', 'NS': '#cccccc'}
    for d in ['NS', 'Up', 'Down']:
        m = df['Direction'] == d
        ax.scatter(df.loc[m, 'logFC'],
                   -np.log10(df.loc[m, 'adj_pvalue'].clip(lower=1e-300)),
                   c=palette[d], s=5, alpha=0.5, label=f'{d} ({m.sum()})')
    # Label top genes
    top = df[df['Direction'] != 'NS'].nsmallest(top_n, 'adj_pvalue')
    for _, r in top.iterrows():
        ax.annotate(r['Gene'],
                    (r['logFC'], -np.log10(max(r['adj_pvalue'], 1e-300))),
                    fontsize=6, alpha=0.8)
    ax.axhline(-np.log10(0.05), color='grey', ls='--', lw=0.5, label='adj.p = 0.05')
    ax.set_xlabel('log₂FC'); ax.set_ylabel('−log₁₀(adj. p-value)')
    ax.set_title(title); ax.legend(fontsize=7, loc='upper right')

# How many panels?
n_panels = 1 + (deg_g1_vs_normal is not None) + (deg_g3_vs_g1 is not None)
fig, axes = plt.subplots(1, n_panels, figsize=(5.5*n_panels, 5))
if n_panels == 1: axes = [axes]

idx = 0
volcano(deg_cancer, 'Tumour vs Normal (all grades)', axes[idx]); idx += 1
if deg_g1_vs_normal is not None:
    volcano(deg_g1_vs_normal, 'Grade 1 vs Normal', axes[idx]); idx += 1
if deg_g3_vs_g1 is not None:
    volcano(deg_g3_vs_g1, 'Grade 3 vs Grade 1', axes[idx])

plt.tight_layout()
plt.savefig('figures/nb2_02_volcanos.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Pathway Enrichment

In [ ]:
# ── 8 · Pathway enrichment via Enrichr ─────────────────────────
import gseapy as gp

def enrichment(gene_list, tag, max_genes=500):
    genes = [str(g) for g in gene_list[:max_genes]]
    if len(genes) < 5:
        print(f'  {tag}: only {len(genes)} genes — skipping'); return None
    print(f'  {tag}: enriching {len(genes)} genes …')
    try:
        enr = gp.enrichr(gene_list=genes,
                         gene_sets=['GO_Biological_Process_2023',
                                    'KEGG_2021_Human'],
                         organism='human', outdir=None, no_plot=True)
        res = enr.results[enr.results['Adjusted P-value'] < 0.05]
        res = res.sort_values('Adjusted P-value')
        res.to_csv(f'results/enrichment_{tag}.csv', index=False)
        n_go   = (res['Gene_set'] == 'GO_Biological_Process_2023').sum()
        n_kegg = (res['Gene_set'] == 'KEGG_2021_Human').sum()
        print(f'    → {n_go} GO-BP + {n_kegg} KEGG significant terms')
        return res
    except Exception as e:
        print(f'    Failed: {e}'); return None

cancer_degs = deg_cancer[deg_cancer['Direction'] != 'NS']['Gene'].tolist()
print('=== Tumour vs Normal ===')
enr_cancer = enrichment(cancer_degs, 'cancer_vs_normal')

if deg_g1_vs_normal is not None:
    g1_degs = deg_g1_vs_normal[deg_g1_vs_normal['Direction'] != 'NS']['Gene'].tolist()
    print('\n=== Grade 1 vs Normal ===')
    enr_g1 = enrichment(g1_degs, 'grade1_vs_normal')

if deg_g3_vs_g1 is not None:
    prog_degs = deg_g3_vs_g1[deg_g3_vs_g1['Direction'] != 'NS']['Gene'].tolist()
    print('\n=== Grade 3 vs Grade 1 ===')
    enr_prog = enrichment(prog_degs, 'grade3_vs_grade1')

In [ ]:
# ── Enrichment bar plots ───────────────────────────────────────

def plot_terms(enr_df, title, gene_set=None, top_n=15):
    if enr_df is None or len(enr_df) == 0:
        print(f'No significant terms for {title}'); return
    df = enr_df.copy()
    if gene_set:
        df = df[df['Gene_set'] == gene_set]
    df = df.head(top_n)
    if len(df) == 0: return
    df['short'] = df['Term'].str[:65]
    fig, ax = plt.subplots(figsize=(10, max(3, len(df)*0.38)))
    ax.barh(range(len(df)), -np.log10(df['Adjusted P-value']),
            color='#e74c3c', edgecolor='black', lw=0.3)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['short'], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('−log₁₀(adj. p-value)'); ax.set_title(title)
    plt.tight_layout()
    fname = title.replace(' ', '_').replace('/', '_')
    plt.savefig(f'figures/nb2_03_{fname}.png', dpi=150, bbox_inches='tight')
    plt.show()

if enr_cancer is not None:
    plot_terms(enr_cancer, 'Cancer vs Normal — GO Biological Process',
              'GO_Biological_Process_2023')
    plot_terms(enr_cancer, 'Cancer vs Normal — KEGG',
              'KEGG_2021_Human')

## 9. Known PMOS–EC Bridge Gene Validation

Check whether known PCOS-to-endometrial-cancer bridge genes are differentially expressed in the TCGA-UCEC data.

In [ ]:
# ── 9 · Known-gene look-up ─────────────────────────────────────

KNOWN_GENES = {
    'ESR1':    'Estrogen receptor α',
    'PGR':     'Progesterone receptor',
    'INSR':    'Insulin receptor',
    'PIK3CA':  'PI3K catalytic subunit',
    'AKT1':    'AKT kinase',
    'PTEN':    'Tumour suppressor',
    'TP53':    'p53',
    'CTNNB1':  'β-catenin',
    'KRAS':    'RAS oncogene',
    'SERPINE1': 'PAI-1 (thrombosis / invasion)',
    'CYP19A1': 'Aromatase',
    'MMP9':    'Matrix metalloproteinase',
    'IL6':     'Interleukin-6',
    'TNF':     'TNFα',
    'VEGFA':   'Angiogenesis',
    'MYD88':   'Innate immune signalling',
    'JUN':     'AP-1 transcription factor',
    'CXCL8':   'IL-8 chemokine',
    'IFNG':    'IFN-γ',
    'IL10':    'Anti-inflammatory cytokine',
}

rows = []
for gene, desc in KNOWN_GENES.items():
    hit = deg_cancer[deg_cancer['Gene'] == gene]
    if len(hit) > 0:
        r = hit.iloc[0]
        rows.append({
            'Gene': gene, 'Role': desc,
            'logFC': f"{r['logFC']:+.3f}",
            'adj_pvalue': f"{r['adj_pvalue']:.2e}",
            'Direction': r['Direction'],
        })
    else:
        rows.append({'Gene': gene, 'Role': desc,
                     'logFC': '—', 'adj_pvalue': '—', 'Direction': 'Not found'})

known_df = pd.DataFrame(rows)
known_df.to_csv('results/known_genes_cancer_check.csv', index=False)

n_found = (known_df['Direction'] != 'Not found').sum()
n_sig   = known_df['Direction'].isin(['Up', 'Down']).sum()
print(f'Known PMOS–EC bridge genes in TCGA-UCEC:')
print(f'  Found: {n_found}/{len(KNOWN_GENES)}   Significant: {n_sig}/{len(KNOWN_GENES)}')
print()
print(known_df.to_string(index=False))

## 10. Save All Outputs

In [ ]:
# ── 10 · Summary ──────────────────────────────────────────────

n_total = (deg_cancer['Direction'] != 'NS').sum()
n_up    = (deg_cancer['Direction'] == 'Up').sum()
n_down  = (deg_cancer['Direction'] == 'Down').sum()

print('═'*60)
print('NOTEBOOK 2 COMPLETE — Endometrial Cancer (TCGA-UCEC)')
print('═'*60)
print(f'\nDataset     : TCGA-UCEC ({len(tumor_ids)} tumour, {len(normal_ids)} normal)')
print(f'Method      : limma eBayes (trend=TRUE) | adj.p < 0.05')
print(f'Genes tested: {len(deg_cancer):,}')
print(f'DEGs        : {n_up} Up + {n_down} Down = {n_total} total')
if deg_g1_vs_normal is not None:
    n = (deg_g1_vs_normal['Direction'] != 'NS').sum()
    print(f'Grade1→Normal : {n} DEGs')
if deg_g3_vs_g1 is not None:
    n = (deg_g3_vs_g1['Direction'] != 'NS').sum()
    print(f'Grade3→Grade1 : {n} DEGs')
print(f'\nKey output  : results/all_genes_limma_tcga_ucec.csv')
print(f'\n→ Next: Notebook 3 (Transition Evidence — hyperplasia vs cancer)')

In [ ]:
# ── (Optional) Download results as ZIP ─────────────────────────
!zip -r nb2_cancer_results.zip results/ figures/
try:
    from google.colab import files
    files.download('nb2_cancer_results.zip')
except Exception:
    print('Download the ZIP from the file browser on the left.')